# Motor de Simulação FP&A - Vértice Retail
Este notebook consolida o pipeline de dados para simulação de cenários operacionais e avaliação de viabilidade financeira (Capex vs Opex).

**Diretrizes de Rigor de KPIs Aplicadas:**
1. **Baseline 2023 Estrito:** Nenhuma receita foi inferida para 2024-2025 via modelos estatísticos/ML. A receita de 2023 é a âncora fixa.
2. **Isolamento de Impacto:** O corte ou injeção de Mídia e as deflexões de Customer Success são tratados em agregações independentes, não afetando diretamente a receita artificialmente, mas sim a *Margem de Contribuição Efetiva por Pedido*.
3. **Métricas Consistentes:** O cálculo de Payback leva em conta o custo de oportunidade anualizado e o investimento inicial (CAPEX).

In [3]:
import pandas as pd
import numpy as np

DATA_ROOT = '../data/'

# Carregamento e higienização dos dados
try:
    vendas = pd.read_csv(DATA_ROOT + 'vendas.csv')
    atend = pd.read_csv(DATA_ROOT + 'atendimento.csv')
    estq = pd.read_csv(DATA_ROOT + 'estoque.csv')
    mkt = pd.read_csv(DATA_ROOT + 'marketing.csv')
    
    vendas['data_pedido'] = pd.to_datetime(vendas['data_pedido'], errors='coerce')
    atend['data_abertura'] = pd.to_datetime(atend['data_abertura'], errors='coerce')
    
    print("Dados carregados com sucesso.")
except FileNotFoundError as e:
    print(f"Erro ao carregar arquivo: {e}\nCertifique-se de ter os CSVs no diretório.")

Dados carregados com sucesso.


## 1. Processamento da Linha de Base (Baseline Run-Rate 2023)
Cálculo rigoroso da base operacional real para ancoragem dos cenários.

In [4]:
# Filtro de 2023
vendas_2023 = vendas[vendas['data_pedido'].dt.year == 2023].copy()

# KPIs Base (KPIs Estritos)
receita_liquida_23 = vendas_2023['receita_liquida'].sum()
pedidos_23 = vendas_2023['order_id'].nunique()
mc_23 = vendas_2023['margem_contribuicao'].sum()
mc_por_pedido_23 = mc_23 / pedidos_23

# KPI de Logística Reversa Base
devolucoes_23 = vendas_2023[vendas_2023['devolvido'] == True]
motivos_alvo = ['Tamanho errado', 'Produto com defeito', 'Atraso na entrega']
custo_frete_reverso_23 = devolucoes_23[devolucoes_23['motivo_devolucao'].isin(motivos_alvo)]['custo_frete'].sum()

print(f"--- KPIs Ancorados 2023 ---")
print(f"Receita Líquida: R$ {receita_liquida_23:,.2f}")
print(f"Total de Pedidos: {pedidos_23}")
print(f"Margem de Contribuição Base: R$ {mc_23:,.2f} (R$ {mc_por_pedido_23:.2f}/pedido)")
print(f"Custo de Frete Reverso Crítico: R$ {custo_frete_reverso_23:,.2f}")

--- KPIs Ancorados 2023 ---
Receita Líquida: R$ 18,086,795.16
Total de Pedidos: 26538
Margem de Contribuição Base: R$ 9,832,931.94 (R$ 370.52/pedido)
Custo de Frete Reverso Crítico: R$ 34,782.66


## 2. Preparação dos Vetores de Simulação (2024-2025)
Extração das variáveis de ineficiência operacional dos anos subsequentes, anualizadas para compatibilidade com o Baseline de 12 meses.

In [5]:
atend_24_25 = atend[atend['data_abertura'].dt.year.isin([2024, 2025])].copy()

# A. Parâmetros de Customer Service (Deflexão)
tkts_onde_pedido = atend_24_25[(atend_24_25['categoria_problema'] == 'Onde está meu pedido?') & 
                               (atend_24_25['canal_entrada'].isin(['E-mail', 'WhatsApp', 'Telefone', 'Reclame Aqui']))]
tkts_onde_ano = len(tkts_onde_pedido) / 2.0
custo_onde_ano = tkts_onde_pedido['custo_operacional_ticket'].sum() / 2.0
tempo_medio_manual = tkts_onde_pedido['tempo_primeira_resposta_minutos'].mean()

cb_time = atend_24_25[(atend_24_25['canal_entrada'] == 'ChatBot') & 
                      (atend_24_25['categoria_problema'] == 'Onde está meu pedido?')]['tempo_primeira_resposta_minutos'].mean()

# B. Parâmetros de Logística Reversa (CS)
tkts_def_tam = atend_24_25[atend_24_25['categoria_problema'].isin(['Defeito', 'Troca de Tamanho'])]
custo_tkts_def_tam_ano = tkts_def_tam['custo_operacional_ticket'].sum() / 2.0

# C. Risco de Ruptura
estq_risco = estq[estq['status_disponibilidade'].isin(['Estoque Crítico', 'Ruptura'])].copy()
giro_23 = vendas_2023.groupby('sku_id').agg(
    qtd_vendida=('quantidade', 'sum'), preco_medio=('preco_unitario', 'mean')
).reset_index()
estq_risco = estq_risco.merge(giro_23, on='sku_id', how='left')
receita_protegida_total = (estq_risco['qtd_vendida'].fillna(0) * estq_risco['preco_medio'].fillna(0)).sum()

## 3. Motor de Simulação Paramétrica
Você pode alterar o parâmetro `capex_implantacao` ou injetar novas variáveis nas chamadas.

In [6]:
def simular_cenario(nome_cenario, taxa_deflexao_chatbot, reducao_devolucoes_perc, capex_implantacao=50000):
    """
    Calcula o retorno financeiro estrito com base na eficiência operacional (Opex).
    """
    # Economia 1: Migração de Tickets (Custo chatbot fixado em R$ 2,00 por atendimento)
    tkts_migrados = tkts_onde_ano * taxa_deflexao_chatbot
    custo_evitado = custo_onde_ano * taxa_deflexao_chatbot
    novo_custo_bot = tkts_migrados * 2.00 
    economia_suporte = custo_evitado - novo_custo_bot
    
    # Impacto no SLA de Resposta
    tempo_medio_ponderado = ((1 - taxa_deflexao_chatbot) * tempo_medio_manual) + (taxa_deflexao_chatbot * cb_time)
    
    # Economia 2: Logística Reversa (Frete + Atendimento de Troca/Defeito)
    economia_frete = custo_frete_reverso_23 * reducao_devolucoes_perc
    economia_tkts = custo_tkts_def_tam_ano * reducao_devolucoes_perc
    economia_log_rev = economia_frete + economia_tkts
    
    # KPI Consolidados
    economia_total_opex = economia_suporte + economia_log_rev
    nova_margem_pedido = (mc_23 + economia_total_opex) / pedidos_23
    
    # Cálculo Rigoroso de Payback (em meses)
    payback_meses = (capex_implantacao / economia_total_opex) * 12 if economia_total_opex > 0 else np.inf
    
    return {
        'Cenário': nome_cenario,
        'Automação (%)': f"{taxa_deflexao_chatbot*100:.0f}%",
        'Redução Dev. (%)': f"{reducao_devolucoes_perc*100:.0f}%",
        'Eco. Suporte (R$)': round(economia_suporte, 2),
        'Eco. Logística Rev. (R$)': round(economia_log_rev, 2),
        'Economia Total Opex (R$)': round(economia_total_opex, 2),
        'SLA Resposta (min)': round(tempo_medio_ponderado, 1),
        'Margem Unitária (R$)': round(nova_margem_pedido, 2),
        'Receita Protegida (R$)': round(receita_protegida_total, 2),
        'Payback (Meses)': round(payback_meses, 1)
    }

## 4. Executar Simulações
Adicione os seus próprios cenários na lista `meus_cenarios` abaixo testando diferentes taxas de deflexão (0 a 1) e reduções de devolução (0 a 1).

In [7]:
meus_cenarios = [
    simular_cenario('Atual / Baseline', 0.0, 0.0),
    simular_cenario('Conservador', 0.50, 0.10),
    simular_cenario('Moderado', 0.75, 0.20),
    simular_cenario('Agressivo', 0.90, 0.30),
    
    # ---- ADICIONE SEUS CENÁRIOS ABAIXO ----
    # Exemplo: Teste de estresse (Cenário Pessimista - 20% deflexão, 5% redução de devolução, CAPEX estourado de 75k)
    simular_cenario('Teste Estresse Usuário', 0.20, 0.05, capex_implantacao=75000)
]

df_resultados = pd.DataFrame(meus_cenarios)
display(df_resultados)

,Cenário,Automação (%),Redução Dev. (%),Eco. Suporte (R$),Eco. Logística Rev. (R$),Economia Total Opex (R$),SLA Resposta (min),Margem Unitária (R$),Receita Protegida (R$),Payback (Meses)
0,Atual / Baseline,0%,0%,0.00,0.00,0.00,171.3,370.52,3058321.11,inf
1,Conservador,50%,10%,22767.75,9319.77,32087.52,86.4,371.73,3058321.11,18.7
2,Moderado,75%,20%,34151.62,18639.53,52791.16,43.9,372.51,3058321.11,11.4
3,Agressivo,90%,30%,40981.95,27959.30,68941.25,18.5,373.12,3058321.11,8.7
4,Teste Estresse Usuário,20%,5%,9107.10,4659.88,13766.98,137.3,371.04,3058321.11,65.4
